# Microfluidic droplet-formation solver

Closed-form Stokes field, Chebyshev verification, and equatorial force balance.

Governing equation:

$$
\frac{1}{r}\frac{\mathrm{d}}{\mathrm{d}r}\left(r\frac{\mathrm{d}u_i}{\mathrm{d}r}\right)
=
\frac{1}{\mu_i}\frac{\partial p_i}{\partial x}
$$

The two pressure gradients are independent and are fixed by $Q_1$ and $Q_2$.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
if not (root / "droplet_formation").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt

from droplet_formation import (
    Channel,
    mlh,
    solve_analytical,
    solve_chebyshev,
    forces_on_droplet,
    critical_diameter,
    equilibrium_diameter,
    real_diameter,
)

ch = Channel()
R, mu1, mu2 = ch.R, ch.mu1, ch.mu2
print(f"R = {R*1e6:.1f} μm,  μ1/μ2 = {mu1/mu2:.3f}")

## Analytical velocity field

Reference case: $Q_1=10\,\mathrm{ml/h}$, $Q_2=1\,\mathrm{ml/h}$, $d_d=0.6\,d_c$.

In [ ]:
Q1, Q2 = mlh(10), mlh(1)
Rd = 0.6 * R
fld = solve_analytical(R, Rd, mu1, mu2, Q1, Q2)
Q1r, Q2r = fld.reconstruct_Q()

print(f"∂p1/∂x = {fld.P1:.4e} Pa/m")
print(f"∂p2/∂x = {fld.P2:.4e} Pa/m")
print(f"U_tp   = {fld.Utp:.5f} m/s")
print(f"Q1, Q2 recovered rel. error {(Q1r-Q1)/Q1:.2e}, {(Q2r-Q2)/Q2:.2e}")
print(f"u1(R) = {float(np.asarray(fld.u1(R))):.2e}")
print(f"interface match: {float(np.asarray(fld.u1(Rd))):.5f} vs {float(np.asarray(fld.u2(Rd))):.5f}")

## Chebyshev collocation

In [ ]:
chb = solve_chebyshev(R, Rd, mu1, mu2, Q1, Q2, N=50)
err1 = np.max(np.abs(chb.u1 - fld.u1(chb.r1)))
err2 = np.max(np.abs(chb.u2[1:] - fld.u2(chb.r2[1:])))
print(f"max |u1| error = {err1:.3e}")
print(f"max |u2| error = {err2:.3e}")

plt.figure(figsize=(6.2, 4.2))
r = np.linspace(1e-9, R, 400)
plt.plot(r * 1e6, fld.u(r), "k-", lw=1.2, label="analytical")
plt.plot(chb.r2 * 1e6, chb.u2, "o", ms=3, label=r"Chebyshev $u_2$")
plt.plot(chb.r1 * 1e6, chb.u1, "o", ms=3, label=r"Chebyshev $u_1$")
plt.axvline(Rd * 1e6, color="r", ls="--", lw=0.8)
plt.xlabel(r"$r$ (μm)")
plt.ylabel(r"$u$ (m/s)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Force balance

$$
F_{\mathrm{tot}}=F_{\Delta p}+F_s-F_r
$$

In [ ]:
ch_sigma = Channel(sigma=2.636e-3)
Q1b, Q2b = mlh(4.456), mlh(0.518)
deq = equilibrium_diameter(ch_sigma, Q1b, Q2b)
dcrit = critical_diameter(ch_sigma.dc, Q1b, Q2b)
print(f"d_eq   = {deq*1e6:.1f} μm" if deq else "d_eq not found")
print(f"d_crit = {dcrit*1e6:.1f} μm")
if deq:
    print(f"d_real = {real_diameter(deq, ch_sigma.dc, Q1b, Q2b)*1e6:.1f} μm")

dds = np.linspace(0.2 * ch_sigma.dc, 0.97 * ch_sigma.dc, 60)
Fr, Fs, Fdp, Ftot = [], [], [], []
for dd in dds:
    f = solve_analytical(ch_sigma.R, 0.5 * dd, ch_sigma.mu1, ch_sigma.mu2, Q1b, Q2b)
    F = forces_on_droplet(f, ch_sigma.sigma)
    Fr.append(F.Fr); Fs.append(F.Fs); Fdp.append(F.Fdp); Ftot.append(F.Ftot)

plt.figure(figsize=(6.4, 4.2))
plt.plot(dds * 1e6, np.array(Fr) * 1e6, label=r"$F_r$")
plt.plot(dds * 1e6, np.array(Fs) * 1e6, label=r"$F_s$")
plt.plot(dds * 1e6, np.array(Fdp) * 1e6, label=r"$F_{\Delta p}$")
plt.plot(dds * 1e6, np.array(Ftot) * 1e6, "k", label=r"$F_{\mathrm{tot}}$")
plt.axhline(0.0, color="k", lw=0.6)
plt.xlabel(r"$d_d$ (μm)")
plt.ylabel(r"Force (μN)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()